<a href="https://colab.research.google.com/github/athirai-s/Maize-RL/blob/master/notebooks/bc_po_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/athirai-s/Maize-RL.git /content/Maize-RL
!git clone https://github.com/Sea-Snell/JaxSeq.git /content/JaxSeq
!apt-get install python3.9 python3.9-distutils python3.9-dev -y
!curl -s https://bootstrap.pypa.io/get-pip.py | python3.9
!python3.9 -m pip install jax[cuda11_pip]==0.4.17 jaxlib==0.4.17 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!python3.9 -m pip install numpy==1.26.4 scipy==1.11.4 chex==0.1.8 optax==0.1.7
!python3.9 -m pip install nvidia-cudnn-cu11==8.6.0.163
!python3.9 -m pip install transformers==4.26.1 tyro wandb flax==0.7.4 tqdm jaxtyping dill cloudpickle
!python3.9 -m pip install --force-reinstall --no-deps jax==0.4.17 jaxlib==0.4.17+cuda11.cudnn86 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!python3.9 -m pip install --force-reinstall --no-deps flax==0.7.4
!python3.9 -m pip install -e /content/JaxSeq --no-deps
!python3.9 -m pip install -e /content/Maize-RL/LMRL-Gym --no-deps
!python3.9 -m pip install redis flask flask-cors frozendict sseclient-py sentencepiece h5py dm-tree einops msgpack gcsfs google-cloud-storage ipython chess nltk rouge-score scikit-image tiktoken==0.5.1 termcolor --ignore-installed blinker
!find /content/JaxSeq /content/Maize-RL -name "*.py" -exec sed -i 's/jnp\.DeviceArray/jax.Array/g' {} \;
!find /content/JaxSeq /content/Maize-RL -name "*.py" -exec sed -i 's/jax\.numpy\.DeviceArray/jax.Array/g' {} \;
!python3.9 -c "import jax; print('JAX:', jax.__version__); print('devices:', jax.devices())"

Cloning into '/content/Maize-RL'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 437 (delta 42), reused 80 (delta 23), pack-reused 321 (from 2)
Receiving objects: 100% (437/437), 51.50 MiB | 45.50 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Cloning into '/content/JaxSeq'...
remote: Enumerating objects: 572, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 572 (delta 9), reused 4 (delta 4), pack-reused 559 (from 1)
Receiving objects: 100% (572/572), 204.95 KiB | 3.20 MiB/s, done.
Resolving deltas: 100% (381/381), done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.9 libpython3.9-dev libpython3.9-minimal libpython3.9-stdlib
  python3.9-lib2to3 python3.9-minimal
Suggested packages:
  python3.9-venv

In [2]:
!nvidia-smi

Wed Apr 22 01:43:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             55W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/data', exist_ok=True)
!wget -q -O /content/data/partially_observed_filtered_maze_data.jsonl https://rail.eecs.berkeley.edu/datasets/rl-llm-bench-dataset/maze/partially_observed_filtered_maze_data.jsonl
!ls -lh /content/data/


Mounted at /content/drive
total 120K
-rw-r--r-- 1 root root 119K Dec 14  2023 partially_observed_filtered_maze_data.jsonl


In [9]:
!rm -rf /usr/local/lib/python3.9/dist-packages/scipy
!rm -rf /usr/local/lib/python3.9/dist-packages/scipy-*.dist-info
!python3.9 -m pip install --no-cache-dir --force-reinstall scipy==1.11.4
!python3.9 -c "import scipy; print('version:', scipy.__version__); print('file:', scipy.__file__); import scipy.linalg; print('tril exists:', hasattr(scipy.linalg, 'tril'))"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/36.6 MB 32.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 26.4 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scipy]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxseq 1.0.0 requires openai==0.27.2, which is not installed.
jaxseq 1.0.0 requires sklearn==0.0, which is not installed.
jaxseq 1.0.0 requires streamlit==1.20.0, which is not installed.
jaxseq 1.0.0 requires torch==1.11, which is not installed.
llm-rl 1.0.0 requires openai==0.27.2, which is not installed.
llm-rl 1.0.0 requires sklearn==0.0, which is not installed.
llm-rl 1.0.0 requires stockfish, which is not installed.
llm-rl 1.0.0 requires streamlit==1.20.0, 

In [15]:
!python3.9 -m pip install --force-reinstall --no-deps ml_dtypes==0.3.2
!python3.9 -c "import jax; print('JAX:', jax.__version__); print('devices:', jax.devices())"


  Using cached ml_dtypes-0.3.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached ml_dtypes-0.3.2-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.2 MB)
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
JAX: 0.4.17
devices: [cuda(id=0)]


In [20]:
!mkdir -p /content/drive/MyDrive/maize_rl_outputs/bc_po
import subprocess
proc = subprocess.Popen(
    ['bash', '-c', '''
cd /content/Maize-RL/LMRL-Gym && python3.9 -m llm_rl_scripts.maze.bc.partially_observed_bc \
    HF gpt2 \
    /content/data/partially_observed_filtered_maze_data.jsonl \
    0.9 \
    --outputs-path=/content/drive/MyDrive/maize_rl_outputs/bc_po/ \
    --exp-name=po_bc_gpt2_small \
    --epochs=30 \
    --lr=1e-4 \
    --train-bsize=16 \
    --grad-accum-steps=8 \
    --max-input-length=1016 \
    --max-output-length=8 \
    --eval-every-steps=256 \
    --save-every-steps=500 \
    --save-at-end \
    --bf16-activations \
    > /content/drive/MyDrive/maize_rl_outputs/bc_po/train.log 2>&1
'''],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True
)
print(f"Training PID: {proc.pid}")


Training PID: 7848


In [45]:
!grep -E "epoch.*step.*train|new best|saving checkpoint" /content/drive/MyDrive/maize_rl_outputs/bc_po/train.log | tail -20
!echo "---"
!ps -p 7848 > /dev/null && echo "Still running" || echo "Process died"

new best model!
saving checkpoint best ...
19it [00:10,  4.17it/s]saving checkpoint step_500 ...
{'epoch': 6, 'step': 512, 'train': {'loss': 32.75271987915039}}
new best model!
saving checkpoint best ...
29it [00:05,  5.24it/s]{'epoch': 9, 'step': 768, 'train': {'loss': 16.443994522094727}}
new best model!
saving checkpoint best ...
39it [00:14,  5.13it/s]saving checkpoint step_1000 ...
{'epoch': 12, 'step': 1024, 'train': {'loss': 5.910614490509033}}
new best model!
saving checkpoint best ...
49it [00:09,  5.12it/s]{'epoch': 15, 'step': 1280, 'train': {'loss': 1.8843963146209717}}
new best model!
saving checkpoint best ...
59it [00:18,  5.11it/s]saving checkpoint step_1500 ...
{'epoch': 18, 'step': 1536, 'train': {'loss': 1.171828269958496}}
new best model!
saving checkpoint best ...
---
Still running


In [46]:
!tail -5 /content/drive/MyDrive/maize_rl_outputs/bc_po/train.log
!echo "---"
!ps aux | grep partially_observed_bc | grep -v grep | awk '{print "PID:", $2, "CPU%:", $3, "MEM%:", $4}'


  File "/content/JaxSeq/JaxSeq/train.py", line 225, in train_loop
    trainer, _, info = trainer.step(
  File "/content/JaxSeq/JaxSeq/models/base_interface.py", line 400, in step
    train_state, loss, logs = self._step(
ValueError: INTERNAL: Failed to execute XLA Runtime executable: run time error: custom call 'xla.gpu.graph.launch' failed: Failed to update gpu graph: Graph update result=kNodeTypeChanged: Failed to update CUDA graph: CUDA_ERROR_GRAPH_EXEC_UPDATE_FAILURE: the graph update was not performed because it included changes which violated constraints specific to instantiated graph update; current profiling annotation: XlaModule:#hlo_module=pjit__step,program_id=42#.
---


In [58]:
%%writefile /tmp/extract.py
import os
import jax
import jax.numpy as jnp
import optax
from transformers import AutoTokenizer
from JaxSeq.models.gpt2.load import load_train_state, ModelLoadMode
from JaxSeq.utils import load_mesh
from JaxSeq.checkpointing import save_pytree

ckpt_dir = '/content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_gpt2_small.2026-04-22-01-59-24.785.e3297e5a3dee11f198d90242ac1c000c/best'

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
mesh = load_mesh((1, 1, -1), ('dp', 'fsdp', 'mp'))

print("Loading train_state from checkpoint...")
train_state, model = load_train_state(
    model_load_mode=ModelLoadMode.TRAIN_STATE,
    model_load_path=ckpt_dir,
    model_dtype=jnp.float32,
    optim_getter=lambda p: optax.adam(1e-4),
    tokenizer=tokenizer,
    mesh=mesh,
    prng_key=jax.random.PRNGKey(0),
    force_pad_embeddings=False,
    params_dtype=jnp.float32,
)
print("Loaded. Saving params.msgpack...")
save_pytree(
    train_state.params,
    os.path.join(ckpt_dir, 'params.msgpack'),
    dtype=jnp.float32,
    sharding=None,
)
print(f"Saved, size: {os.path.getsize(os.path.join(ckpt_dir, 'params.msgpack'))}")


Overwriting /tmp/extract.py


In [63]:
!cat /tmp/extract.py | grep model_load_mode


    model_load_mode=ModelLoadMode.TRAIN_STATE,


In [64]:
%%writefile /tmp/extract.py
import os
import jax
import jax.numpy as jnp
import optax
from transformers import AutoTokenizer
from JaxSeq.models.gpt2.load import load_train_state, ModelLoadMode
from JaxSeq.utils import load_mesh
from JaxSeq.checkpointing import save_pytree

ckpt_dir = '/content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_gpt2_small.2026-04-22-01-59-24.785.e3297e5a3dee11f198d90242ac1c000c/best'

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
mesh = load_mesh((1, 1, -1), ('dp', 'fsdp', 'mp'))

print("Loading train_state (params only) from checkpoint...")
train_state, model = load_train_state(
    model_load_mode=ModelLoadMode.TRAIN_STATE_PARAMS,
    model_load_path=ckpt_dir,
    model_dtype=jnp.float32,
    optim_getter=lambda p: optax.adam(1e-4),
    tokenizer=tokenizer,
    mesh=mesh,
    prng_key=jax.random.PRNGKey(0),
    force_pad_embeddings=False,
    params_dtype=jnp.float32,
)
print("Loaded. Saving params.msgpack...")
save_pytree(
    train_state.params,
    os.path.join(ckpt_dir, 'params.msgpack'),
    dtype=jnp.float32,
    sharding=None,
)
print(f"Saved, size: {os.path.getsize(os.path.join(ckpt_dir, 'params.msgpack'))}")


Overwriting /tmp/extract.py


In [65]:
!cat /tmp/extract.py | grep model_load_mode

    model_load_mode=ModelLoadMode.TRAIN_STATE_PARAMS,


In [66]:
!python3.9 /tmp/extract.py

/usr/local/lib/python3.9/dist-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/usr/local/lib/python3.9/dist-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/usr/local/lib/python3.9/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new down

In [51]:
!mkdir -p /content/outputs/eval_bc_po
!cd /content/Maize-RL/LMRL-Gym && XLA_FLAGS="--xla_gpu_graph_level=0" python3.9 -m llm_rl_scripts.maze.bc.eval_bc \
    PARAMS /content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_gpt2_small.2026-04-22-01-59-24.785.e3297e5a3dee11f198d90242ac1c000c/best \
    --outputs-path=/content/outputs/eval_bc_po/ \
    --no-fully-observed \
    --policy-n-rollouts=32 \
    --policy-bsize=1 \
    --policy-max-input-length=1016 \
    --policy-max-output-length=8 \
    --maze-last-k=40


/usr/local/lib/python3.9/dist-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/usr/local/lib/python3.9/dist-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
{'model_load_mode': <ModelLoadMode.PARAMS: 'params'>, 'model_load_path': '/content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_gpt2_small.2026-04-22-01-59-24.785.e3297e5a3dee11f198d90242ac1c000c/best', 'data_mesh_shape': 1, 'fsdp_mes

In [67]:
import subprocess, os
env = os.environ.copy()
env['XLA_FLAGS'] = '--xla_gpu_graph_level=0'
proc = subprocess.Popen(['bash', '-c', '''
cd /content/Maize-RL/LMRL-Gym && python3.9 -m llm_rl_scripts.maze.bc.partially_observed_bc \
    HF gpt2 \
    /content/data/partially_observed_filtered_maze_data.jsonl \
    0.9 \
    --outputs-path=/content/drive/MyDrive/maize_rl_outputs/bc_po/ \
    --exp-name=po_bc_v2 \
    --epochs=20 \
    --lr=1e-4 \
    --train-bsize=16 \
    --grad-accum-steps=8 \
    --max-input-length=1016 \
    --max-output-length=8 \
    --eval-every-steps=256 \
    --save-every-steps=500 \
    --save-at-end \
    --no-save-train-state \
    --bf16-activations \
    > /content/drive/MyDrive/maize_rl_outputs/bc_po/train_v2.log 2>&1
'''], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
print(f"PID: {proc.pid}")


PID: 26147


In [75]:
!grep -E "epoch.*step.*train|new best|saving checkpoint" /content/drive/MyDrive/maize_rl_outputs/bc_po/train_v2.log | tail -15
!echo "---"
!ps -p 26147 > /dev/null && echo "Still running" || echo "Process died"


saving checkpoint best ...
39it [00:09,  5.08it/s]saving checkpoint step_1000 ...
{'epoch': 12, 'step': 1024, 'train': {'loss': 5.876514911651611}}
new best model!
saving checkpoint best ...
49it [00:09,  5.09it/s]{'epoch': 15, 'step': 1280, 'train': {'loss': 1.849159598350525}}
new best model!
saving checkpoint best ...
59it [00:13,  4.98it/s]saving checkpoint step_1500 ...
{'epoch': 18, 'step': 1536, 'train': {'loss': 1.1567518711090088}}
new best model!
saving checkpoint best ...
new best model!
saving checkpoint best ...
saving checkpoint last ...
---
Still running


In [76]:
!ls -lh /content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_v2*/
!echo "---"
!ls -lh /content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_v2*/best/


total 36K
drwx------ 2 root root 4.0K Apr 22 03:24 best
-rw------- 1 root root  14K Apr 22 03:07 config.py
-rw------- 1 root root 1.2K Apr 22 03:07 input_args.pkl
drwx------ 2 root root 4.0K Apr 22 03:24 last
drwx------ 2 root root 4.0K Apr 22 03:16 step_1000
drwx------ 2 root root 4.0K Apr 22 03:21 step_1500
drwx------ 2 root root 4.0K Apr 22 03:12 step_500
---
total 260M
-rw------- 1 root root  990 Apr 22 03:24 config.json
-rw------- 1 root root  508 Apr 22 03:24 loop_state.pkl
-rw------- 1 root root 260M Apr 22 03:24 params.msgpack


In [77]:
!ls -d /content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_v2*/best


/content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_v2.2026-04-22-03-07-34.470.68cde2863df811f1a4550242ac1c000c/best


In [78]:
!mkdir -p /content/outputs/eval_bc_po_v2
!cd /content/Maize-RL/LMRL-Gym && XLA_FLAGS="--xla_gpu_graph_level=0" python3.9 -m llm_rl_scripts.maze.bc.eval_bc \
    PARAMS /content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_v2.2026-04-22-03-07-34.470.68cde2863df811f1a4550242ac1c000c/best \
    --outputs-path=/content/outputs/eval_bc_po_v2/ \
    --no-fully-observed \
    --policy-n-rollouts=32 \
    --policy-bsize=1 \
    --policy-max-input-length=1016 \
    --policy-max-output-length=8 \
    --maze-last-k=40


/usr/local/lib/python3.9/dist-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/usr/local/lib/python3.9/dist-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
{'model_load_mode': <ModelLoadMode.PARAMS: 'params'>, 'model_load_path': '/content/drive/MyDrive/maize_rl_outputs/bc_po/po_bc_v2.2026-04-22-03-07-34.470.68cde2863df811f1a4550242ac1c000c/best', 'data_mesh_shape': 1, 'fsdp_mesh_shape'